# Pattern 08: Metadata + self-query

Follows this repo's mandatory 8-section notebook template -- this is one of "the 10 patterns."

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`** for both the embedding and
generation/filter-extraction steps.

**What's genuinely demonstrated below, and what isn't:** `result.filter_accuracy` is populated
(non-`None`) for the FIRST time in this project in section 5's output below -- that MECHANISM is
real and working (`evals/run.py`'s `filter_accuracy()` call has existed since before this pattern but had no
consumer until this pattern). The NUMBER itself isn't meaningful under mock, though: `MockLLM`
returns the same canned filter-extraction response for every question, so it isn't actually reading
each question to decide what filter to extract. Section 7 (real retrieval-quality findings) is
PENDING, same as every other real-key-blocked pattern.


## Reproducibility header

In [1]:
import platform
import subprocess
import sys

import numpy
import openai

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: d8337d8fafa765c37169264abfbf530e3b1643e6


## Setup (loaded once, used by every section below)

In [2]:
import os

os.environ.setdefault("RAG_RECIPES_LLM", "mock")

from evals.run import load_corpus_by_id, load_qa_set, run_pattern
from recipes.llm import MockLLM, get_llm

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
llm = get_llm()  # used for generation (recipe_fn's own LLM calls)

# Judging needs its own backend: under a real API key this is the same
# real model, but under mock, `llm`'s canned generation text isn't valid
# JSON, and the judge prompts require JSON output. A separate MockLLM
# here demonstrates a clean, illustrative run instead of every question
# correctly (but noisily) failing to parse -- see evals/judges.py's
# JudgeParseError and evals/run.py's per-question error isolation.
if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm

from recipes.embeddings import get_embedder

embedder = get_embedder()


## 1. What this pattern does

Self-query extracts a structured metadata filter (e.g. `{"category": "cs.LG"}`) from the question via
an LLM call (`prompts/self_query_prompt.txt`), BEFORE retrieval. The corpus is ranked by dense
similarity as usual, then restricted to chunks matching the extracted filter, then sliced to top-k.
If no filter is extracted (or extraction fails), retrieval silently falls back to searching the
whole corpus -- no special-case code needed, since an empty filter matches everything by
construction.


## 2. When to use it

- Your corpus has real structured metadata (author, year, category, section) that questions
  reference explicitly ("among the 2024 papers on X...")
- You want filter-then-rank precision rather than hoping semantic similarity alone surfaces the
  right subset
- You can afford one extra LLM call per query for filter extraction


## 3. When NOT to use it

- Your corpus has no meaningful metadata structure, or questions never reference it
- The filter-extraction LLM call can hallucinate a filter value that matches zero chunks -- this
  pattern degrades gracefully (falls back to an empty result set, not a crash), but that's still a
  worse outcome than skipping self-query for corpora where extraction accuracy is unreliable
- Latency/cost budget can't absorb the extra LLM call before retrieval even starts


## 4. Implementation

In [3]:
from recipes.self_query import make_retrieve_and_answer

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

# Try it on one question directly.
sample = retrieve_and_answer("Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?", k=3)
print("retrieved:", sample.retrieved_chunk_ids)
print("extracted_filter:", sample.extracted_filter)
print("answer:", sample.answer)


retrieved: ['arxiv:2601.00907#2', 'arxiv:2601.00907#0', 'arxiv:2601.00907#1']
extracted_filter: {'category': 'cs.LG'}
answer: The paper titled "Placenta Accreta Spectrum Detection using Multimodal Deep Learning" addresses diagnosing a pregnancy complication using deep learning. It focuses on Placenta Accreta Spectrum (PAS), a life-threatening obstetric condition, and develops a multimodal deep learning framework that integrates 3D MRI and 2D Ultrasound scans to improve early and accurate prenatal diagnosis. The study uses unimodal feature extractors (3D DenseNet121-Vision Transformer for MRI and 2D ResNet50 for US) and demonstrates that the multimodal fusion model outperforms unimodal models in accuracy and AUC, showing strong potential to enhance prenatal risk assessment and improve patient outcomes [arxiv:2601.00907#0, #2].


## 5. Run on our eval set

In [4]:
pattern_fn = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

result = run_pattern(
    recipe_fn=pattern_fn,
    qa_set=qa_set,
    corpus_by_id=corpus_by_id,
    llm=judge_llm,
    pattern_name="08_self_query",
    judges_enabled=True,
)
print()
print(f"filter_accuracy populated (not None): {result.filter_accuracy is not None}")


=== 08_self_query (n=18) ===
  hit@3: 1.000  [95% CI 1.000, 1.000]
  hit@10: 1.000  [95% CI 1.000, 1.000]
  mrr: 0.889  [95% CI 0.778, 0.972]
  faithfulness: 0.611  [95% CI 0.389, 0.833]
  answer_relevance: 1.000  [95% CI 1.000, 1.000]
  citation_accuracy: 0.671  [95% CI 0.556, 0.782]
  filter_accuracy: 1.000  [95% CI 1.000, 1.000]
  p50_latency_ms: 4032.4
  p95_latency_ms: 6650.9
  usd_per_query: $0.00683
  eval_usd: $0.1229

filter_accuracy populated (not None): True


## 6. Example query walkthrough

One example per eval-set category, showing the extracted filter, retrieved chunks, and the (mocked)
final answer.

In [5]:
examples = {
    "keyword": "What does PCEval stand for?",
    "paraphrase": "Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?",
    "multi_hop": "The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?",
    "filter": "Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?",
}

for category, question in examples.items():
    result = retrieve_and_answer(question, k=3)
    print(f"--- {category} ---")
    print(f"Q: {question}")
    print(f"extracted_filter: {result.extracted_filter}")
    print(f"Retrieved: {result.retrieved_chunk_ids}")
    print(f"A: {result.answer}")
    print()


--- keyword ---
Q: What does PCEval stand for?
extracted_filter: {}
Retrieved: ['arxiv:2601.02404#0', 'arxiv:2601.02404#2', 'arxiv:2601.00138#0']
A: PCEVAL stands for Physical Computing Evaluation. It is a benchmark designed for fully automatic evaluation of the capabilities of Large Language Models (LLMs) in both the logical and physical aspects of physical computing projects, particularly involving hardware implementation constraints in simulation environments [arxiv:2601.02404#0].



--- paraphrase ---
Q: Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?
extracted_filter: {}
Retrieved: ['arxiv:2601.00090#0', 'arxiv:2601.00090#1', 'arxiv:2601.00090#2']
A: Repeated image generations from the same text prompt in diffusion models end up looking very similar due to a phenomenon called mode collapse. This occurs because the models tend to produce nearly identical outputs across different random seeds when given the same prompt, limiting the diversity of generated images. The initial noise inputs play a crucial role in this, and without proper variation or optimization, the model repeatedly converges to similar results. This issue has been observed in contemporary text-to-image models like Stable Diffusion SDXL-Turbo and Flux.1, where sampling multiple images with a fixed prompt yields surprisingly little visual variation [arxiv:2601.00090#0].



--- multi_hop ---
Q: The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?
extracted_filter: {}
Retrieved: ['arxiv:2601.00129#0', 'arxiv:2601.00130#2', 'arxiv:2601.00129#2']
A: The two photonics-AI-systems papers focus on complementary aspects of the challenge of realizing large-scale photonics-empowered AI systems:

1. The first paper ([arxiv:2601.00129]) emphasizes the need for **physical design automation and system-algorithm co-exploration**. It highlights the importance of scalable, manufacturable photonic and electronic-photonic integrated circuits (PICs/EPICs) and the integration of physical non-idealities, control/calibration limits, and packaging/interface costs into architectural and learning algorithm design. This paper presents a cross-layer toolchain including SimPhony, ADEPT, Apollo, and LiDAR to bridge physical device constraints and system-level architectural decisions, enabling a physically 

--- filter ---
Q: Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?
extracted_filter: {'category': 'cs.LG'}
Retrieved: ['arxiv:2601.00907#2', 'arxiv:2601.00907#0', 'arxiv:2601.00907#1']
A: The paper titled "Placenta Accreta Spectrum Detection using Multimodal Deep Learning" addresses diagnosing a pregnancy complication using deep learning. It focuses on Placenta Accreta Spectrum (PAS), a life-threatening obstetric condition, and develops a multimodal deep learning framework that integrates 3D MRI and 2D Ultrasound scans to improve early and accurate prenatal diagnosis. The model uses a 3D DenseNet121-Vision Transformer for MRI and a 2D ResNet50 for US, achieving superior diagnostic performance compared to unimodal models [arxiv:2601.00907#0, #1, #2].



## 7. Where this pattern FAILS

**PENDING: real findings from a one-off real-embeddings run.** A real `OPENAI_API_KEY` was not yet
available in the environment when this notebook was authored. This section will be replaced with a
static table of genuine hit@k/mrr failures (matching the format used in `02_bm25.ipynb`/
`04_rerank.ipynb` section 7), computed via a real, uncommitted exploratory run once a key is
available. The claim will be labeled with the date it was run
and its actual dollar cost, and will not be presented as live-executed cell output, to avoid
implying a mock re-run reproduces it (see this notebook's top-of-file disclaimer).


## 8. Copy-paste snippet

Meant for pasting into your own project, not executed as a cell in this notebook.

```python
"""Minimal self-query retrieval + generation, no eval harness."""
from recipes.embeddings import get_embedder
from recipes.self_query import make_retrieve_and_answer
from recipes.llm import get_llm

corpus_by_id = {}  # {chunk_id: {"category": ..., "text": ..., ...}, ...} -- fill in your own chunks
embedder = get_embedder()
llm = get_llm()

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)
result = retrieve_and_answer("your question here", k=5)
print(result.answer, result.extracted_filter)
```
